# X-DETR — Colab Free (T4) training
Train the custom X-ray detector on OPIXray with per-epoch Google-Drive checkpointing so a
disconnect/restart resumes automatically. Runtime > Change runtime type > **T4 GPU**.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 1. Mount Drive
Everything (project code, dataset, checkpoints) lives under `MyDrive/xray/` so it survives
disconnects. Run this once.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/xray', exist_ok=True)

## 2. Get the project onto Drive — pick ONE

**Option A — upload a zip of your local `x-ray/` folder (no GitHub needed).**
Locally, run (excludes the dataset and any local run outputs so the zip stays small):
```bash
cd ~/Desktop && zip -r x-ray.zip x-ray -x "x-ray/data/OPIXray/*" -x "x-ray/runs/*"
```
Then run the cell below and pick `x-ray.zip` in the upload dialog.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # choose x-ray.zip
zpath = next(iter(uploaded))
with zipfile.ZipFile(zpath) as z:
    z.extractall('/content/drive/MyDrive/xray')
os.remove(zpath)
print('extracted to /content/drive/MyDrive/xray/x-ray')

**Option B — `git clone` your fork instead** (skip the upload cell above if you use this):
```python
!git clone https://github.com/<you>/x-ray.git /content/drive/MyDrive/xray/x-ray
```

In [ ]:
PROJ = '/content/drive/MyDrive/xray/x-ray'   # <-- adjust if you cloned elsewhere
os.chdir(PROJ); print('cwd:', os.getcwd())
!pip -q install scipy pyyaml gradio && echo done  # torch/torchvision preinstalled on Colab

## 3. Wiring sanity (no data needed) — should print PASS
This validates the model/matcher/loss end-to-end before you touch the dataset.

In [ ]:
!python -m scripts.overfit --config configs/xdetr_opixray.yaml --synthetic --iters 60

## 4. Get OPIXray onto Drive
OPIXray requires a signed academic agreement (see `scripts/download_opixray.md` for the
request process) — it can't be auto-downloaded. Once you have the unzipped folder:

* **Small enough to upload from your machine?** Use a `files.upload()` cell like the one
  above (zip it first), extracting to `/content/drive/MyDrive/xray/OPIXray`.
* **Already have it in Google Drive** (e.g. shared via the author's Drive link)? Use
  Colab's Drive file browser (left sidebar) to copy/move it under `MyDrive/xray/OPIXray`,
  or `!cp -r` it there.

Expected layout: `MyDrive/xray/OPIXray/{train,test}/...` — see `scripts/download_opixray.md`.

In [ ]:
DATA = '/content/drive/MyDrive/xray/OPIXray'
assert os.path.isdir(DATA), f'{DATA} not found yet — finish step 4 first'
!python -m scripts.sanity_data --config configs/xdetr_opixray.yaml \
    --set dataset.root=$DATA --n 6 --out assets/sanity.png
from IPython.display import Image as IPyImage; IPyImage('assets/sanity.png')

## 5. Overfit 10 real images (boxes should visually snap) — correctness on real data

In [ ]:
!python -m scripts.overfit --config configs/xdetr_opixray.yaml \
    --set dataset.root=$DATA --n 10 --iters 300

## 6. Train (T4). Checkpoints go to Drive every epoch
Re-running this cell after a disconnect **resumes automatically** from `last.pth`.

In [ ]:
OUT  = '/content/drive/MyDrive/xray/runs/opixray_xdetr'
!python -m engine.train --config configs/xdetr_opixray.yaml \
    --set dataset.root=$DATA training.output_dir=$OUT training.epochs=50 \
          model.dec_layers=6 model.num_queries=300

## 7. Evaluate (per-class AP, occlusion OL1/2/3, ECE)

In [ ]:
!python -m engine.evaluate --config configs/xdetr_opixray.yaml \
    --weights $OUT/last.pth --set dataset.root=$DATA training.output_dir=$OUT

## 8. Generate the visualization galleries

In [ ]:
!python -m scripts.gallery_batch --config configs/xdetr_opixray.yaml \
    --weights $OUT/last.pth --set dataset.root=$DATA --n 12 --out assets/galleries --score 0.3
from IPython.display import Image as IPyImage, display
import glob
for p in sorted(glob.glob('assets/galleries/*.png'))[:4]:
    display(IPyImage(p))

## 9. (Optional) Gradio demo with a public link
`--share` gives you a temporary public URL since Colab has no local browser.

In [ ]:
!python app/gradio_demo.py --config configs/xdetr_opixray.yaml --weights $OUT/last.pth --share \
    --set dataset.root=$DATA